Tiền xử lý từng file data

In [3]:
import pandas as pd
import numpy as np
import os
import gc

# Danh sách các cấu hình file đầu vào và đầu ra
file_configs = [
    {"input": "2019-Nov.csv", "output": "2019-Nov-Cleaned.csv"}
]

CURRENT_FILE_INDEX = 0 

# Lấy cấu hình hiện tại
config = file_configs[CURRENT_FILE_INDEX]
input_file = config["input"]
output_file = config["output"]

os.makedirs('cleaned', exist_ok=True)
print(f"=== ĐÃ CẤU HÌNH: Sẵn sàng tiền xử lý file: {input_file} ===")

=== ĐÃ CẤU HÌNH: Sẵn sàng tiền xử lý file: 2019-Nov.csv ===


In [4]:
print(f"Đang đọc dữ liệu từ file: {input_file}...")
df = pd.read_csv(input_file)
print(f"Tải thành công! Kích thước ban đầu của file: {df.shape}")
df.head()

Đang đọc dữ liệu từ file: 2019-Nov.csv...
Tải thành công! Kích thước ban đầu của file: (67501979, 9)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [5]:
print("Đang kiểm tra dữ liệu trùng lặp...")
duplicates = df.duplicated().sum()

if duplicates > 0:
    df = df.drop_duplicates(keep='first')
    print(f"-> Đã xóa {duplicates} dòng trùng lặp.")
else:
    print("-> Không có dòng trùng lặp.")
    
print(f"Kích thước dữ liệu hiện tại: {df.shape}")

Đang kiểm tra dữ liệu trùng lặp...
-> Đã xóa 100519 dòng trùng lặp.
Kích thước dữ liệu hiện tại: (67401460, 9)


In [6]:
time_col = 'event_time' 

if time_col in df.columns:
    print(f"Đang chuyển đổi cột '{time_col}' sang Datetime...")
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    print("-> Hoàn tất chuyển đổi định dạng thời gian.")
else:
    print(f"Không tìm thấy cột '{time_col}'. Các cột hiện có: {df.columns.tolist()}")

Đang chuyển đổi cột 'event_time' sang Datetime...
-> Hoàn tất chuyển đổi định dạng thời gian.


In [7]:
# Kiểm tra tỷ lệ missing
missing_percent = (df.isnull().sum() / len(df)) * 100
has_missing = missing_percent[missing_percent > 0]

if not has_missing.empty:
    print("Tỷ lệ dữ liệu thiếu (%):\n", has_missing)
    
    # Xử lý missing cho cột dạng chuỗi (Categorical)
    categorical_cols = df.select_dtypes(include=['object']).columns
    df[categorical_cols] = df[categorical_cols].fillna('Unknown')
    
    # Xử lý missing cho cột dạng số (Numeric) bằng giá trị trung vị (Median)
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())
        
    print("\n-> Đã xử lý xong toàn bộ dữ liệu thiếu.")
else:
    print("-> Bộ dữ liệu không có giá trị thiếu.")

Tỷ lệ dữ liệu thiếu (%):
 category_code    32.449480
brand            13.671824
user_session      0.000015
dtype: float64

-> Đã xử lý xong toàn bộ dữ liệu thiếu.


In [8]:
target_col = 'price'

if target_col in df.columns:
    outlier_count = (df[target_col] < 0).sum()
    if outlier_count > 0:
        df = df[df[target_col] >= 0]
        print(f"-> Đã xóa {outlier_count} dòng có giá trị '{target_col}' bị âm.")
    else:
        print(f"-> Cột '{target_col}' hợp lệ (không có giá trị âm).")

-> Cột 'price' hợp lệ (không có giá trị âm).


In [9]:
print(f"Đang lưu dữ liệu đã tiền xử lý ra file: {output_file}...")
df.to_csv(output_file, index=False)
print("-> Lưu file thành công!")
print(f"Kích thước bộ dữ liệu cuối cùng của file này: {df.shape}")

# Tiến hành xóa DataFrame và ép Python giải phóng RAM ngay lập tức
del df
gc.collect()
print("=== ĐÃ GIẢI PHÓNG RAM! ===")

Đang lưu dữ liệu đã tiền xử lý ra file: 2019-Nov-Cleaned.csv...
-> Lưu file thành công!
Kích thước bộ dữ liệu cuối cùng của file này: (67401460, 9)
=== ĐÃ GIẢI PHÓNG RAM! ===
